# McDonald-Dunn Research Forest: Forest Recovery from Active Timber Management 1985–2025
### Using USFS Landscape Change Monitoring System (LCMS) in Google Earth Engine

---

**Research Question:** How do actively managed compartments at the McDonald-Dunn Research Forest cycle through LCMS-detectable succession stages, and can 40 years of LCMS data capture the full harvest-to-recovery arc for a commercially managed Oregon Coast Range forest?

**Dataset:** [LCMS v2025-11](https://developers.google.com/earth-engine/datasets/catalog/projects_gtac-data-publish_assets_LCMS_Product_Version_2025-11) — USFS/GTAC annual land cover, land use, and change maps at 30 m, 1985–2025 (CONUS + SE Alaska).

LCMS produces three annual thematic products:
| Band | What it shows |
|------|--------------|
| `Land_Cover` | What is on the ground (Trees, Shrubs, Grass, Barren, Water, etc.) |
| `Land_Use` | How the land is used (Forest, Agriculture, Developed, Rangeland, etc.) |
| `Change` | What changed and how (Tree Removal, Successional Growth, Wildfire, Stable, etc.) |

**Why the McDonald-Dunn Research Forest?**  
The [McDonald-Dunn Research Forest](https://cf.forestry.oregonstate.edu/our-forests/mcdonald-and-dunn-forests) is an 11,500-acre teaching and demonstration forest operated by Oregon State University's College of Forestry, located approximately 15 minutes north of the OSU Corvallis campus in the Oregon Coast Range foothills. Unlike LTER sites focused on ecological mechanics, McDonald-Dunn is a **financially self-sustaining, actively managed** forest — timber harvests are conducted throughout the LCMS period on a rotation cycle, making it an ideal site for studying how LCMS captures commercial harvest events and the subsequent recovery arc in a Coast Range Douglas-fir setting.

Key characteristics relevant to LCMS analysis:
- **Continuous active management**: Clearcut, shelterwood, seed-tree, and selection harvests documented throughout 1985–2025
- **Multiple rotation cohorts**: Stands span from early 1950s OSU plantations to harvests within the last decade, creating a rich chronosequence
- **Complex landscape mosaic**: The constant rotation creates a highly dynamic patchwork — in sharp contrast to LTER sites managed for long-term stability
- **Oregon Coast Range Douglas-fir type**: Lower elevation than the western Cascades, with faster early seral transitions due to warmer temperatures and abundant moisture
- **Layered disturbance history**: From Kalapuya burning, to late-1800s settler harvest, to WWII military use (Dunn Forest / Camp Adair), to modern OSU silvicultural research

LCMS covers 1985–2025, which means:
- Areas harvested **before 1985** appear at the *start* of the record already in mid-recovery (older OSU plantations established in the 1950s–1960s)
- Areas harvested **during 1985–2025** show their full post-harvest trajectory captured in LCMS
- Long-uncut old-growth remnant patches provide stable reference pixels throughout

**Prerequisites**
- A Google Earth Engine (GEE) account — [sign up here](https://earthengine.google.com/signup/)
- A GEE Cloud project ID — [create one here](https://console.cloud.google.com/projectcreate)
- Python ≥ 3.9 with `earthengine-api` and `geeViz` installed:
  ```bash
  pip install earthengine-api geeViz
  ```

> 💡 **Workshop note:** This notebook is designed to run sequentially top-to-bottom. All cells are self-documenting. Run `Kernel → Restart & Run All` for a clean start.

## 1 · Setup — Imports and Authentication

In [1]:
import os, ee
from IPython.display import display, HTML

# ── Authentication ────────────────────────────────────────────────────────────
# Run this once per machine/account to store credentials locally.
# After that, comment it out and just call ee.Initialize() below.
#ee.Authenticate()

# ── Initialization ────────────────────────────────────────────────────────────
# ee.Initialize MUST come before geeViz imports so the eeAuth proxy starts
# with the correct project. Importing geeViz first causes it to spin up the
# proxy with whatever cached credentials it finds (potentially from another project).
# Replace 'your-project-id' with your GEE Cloud project ID.
EE_PROJECT = 'rcr-gee'
ee.Initialize(project=EE_PROJECT)

# ── Force in-process HTTP server ──────────────────────────────────────────────
# Without this, geeViz spawns a *detached* eeAuth subprocess from whatever
# Python is on the system PATH. That subprocess serves its own copy of the
# geeViz package directory. The notebook's venv writes run_geeViz.js to the
# venv copy, but the detached process serves the system-Python copy — so the
# browser always fetches stale layer state (e.g. SRTM from a previous run).
# "auto" keeps the HTTP server in-process (same venv, same file paths).
os.environ['GEEVIZ_EEAUTH_MODE'] = 'auto'

# ── geeViz imports ────────────────────────────────────────────────────────────
import geeViz.getImagesLib as gil
import geeViz.geeView
import geeViz.getSummaryAreasLib as sal
from geeViz.outputLib import charts as cl

# -- Set up geeViz Map object --------------------------------------------------
Map = gil.Map  # assign geeViz Map object to variable for convenience
Map.port = 8080
Map.project = EE_PROJECT
Map.clearMap()

# ── Test Earth Engine connection ------------------------------------------------
test = ee.Image(1).getInfo()
print(test, '\n Earth Engine initialized successfully.')


{'type': 'Image', 'bands': [{'id': 'constant', 'data_type': {'type': 'PixelType', 'precision': 'int', 'min': 1, 'max': 1}, 'crs': 'EPSG:4326', 'crs_transform': [1, 0, 0, 0, 1, 0]}]} 
 Earth Engine initialized successfully.


## 2 · Study Area — McDonald-Dunn Research Forest

The [McDonald-Dunn Research Forest](https://cf.forestry.oregonstate.edu/our-forests/mcdonald-and-dunn-forests) encompasses approximately 11,500 acres (4,650 ha) in the Oregon Coast Range foothills, roughly 10 miles north of the OSU Corvallis campus. The combined forest comprises two blocks with distinct histories:

- **McDonald Forest** (~5,000 acres, southern block): acquired beginning in 1926 with donations from Mary McDonald; first harvested by Euro-American settlers in the late 1800s; now carries multiple generations of OSU-managed Douglas-fir plantations
- **Dunn Forest** (~6,500 acres, northern block): former Camp Adair WWII military training ground, acquired by OSU in 1947; prior agricultural and military land use created a complex early land-cover mosaic; bullets can occasionally still be found in older trees

The forest's actively managed history creates a highly dynamic landscape well suited to LCMS analysis:

| Period | Management Activity | LCMS Visibility |
|--------|--------------------|----|
| Pre-1985 | Multiple rotation cycles since OSU acquisition (~1930s onward); early settler harvest; Dunn Forest recovery from WWII-era land use | Mixed-age plantations at varying recovery stages at LCMS start |
| 1985–1994 | Active commercial timber harvest on rotation across numerous compartments; diverse silvicultural treatments for teaching | Tree Removal and Successional Growth signals across multiple compartments per year |
| 1994–2010 | Continued active harvest under updated management plans — unlike adjacent USFS lands, **not** affected by the 1994 Northwest Forest Plan | Ongoing Tree Removal events; mid-rotation stands entering crown closure |
| 2010–2025 | Recent clearcut, shelterwood, and partial harvests; ongoing active management | Tree Removal near end of record; early-seral Shrub/Grass visible; Trees recovery rising |

Because the forest is continuously and actively managed, Tree Removal events should appear somewhere in the study area in nearly every year of the LCMS record — a pattern not visible at LTER or wilderness sites.

> 📍 **Polygon note:** The harvest unit boundaries used in this notebook are approximate. For analysis using verified boundaries, contact the OSU Research Forest Business Office at Peavy Arboretum or consult the [2025 Forest Management Plan](https://cf.forestry.oregonstate.edu/our-forests/2025-mcdonald-dunn-forest-plan).

In [10]:
# -- Study date range: ----------------------------------------------------------
# LCMS data extend from 1984 through 2025. The 2025 collection is the most recent.
START_YEAR = 1984
END_YEAR   = 2024

# ── Primary study area: McDonald-Dunn Research Forest bounding box ────────────
# Combined forest is ~11,500 acres (~46.5 km²); this bbox covers both the
# McDonald (southern) and Dunn (northern) blocks with a small context buffer.
# Forest center: ~44.70 N, 123.34 W — approximately 10 miles north of OSU Corvallis.
study_area = ee.Geometry.BBox(-123.46, 44.60, -123.22, 44.82)

# ── Option B: McDonald Forest block only (southern portion, ~5,000 acres) ─────
# Uncomment to restrict analysis to the original McDonald acquisition area.
# study_area = ee.Geometry.BBox(-123.42, 44.60, -123.27, 44.72)

# -- Print study area info ------------------------------------------------------
print("Study date range:", START_YEAR, "to", END_YEAR)
print('Study area type:', study_area.getInfo()['type'])
area_km2 = study_area.area(maxError=500).divide(1e6).getInfo()
print(f'Bounding box area : {area_km2:,.0f} km2  (McDonald-Dunn combined forest ~46.5 km2; bbox includes surrounding context)')


Study date range: 1984 to 2024
Study area type: Polygon
Bounding box area : 464 km2  (McDonald-Dunn combined forest ~46.5 km2; bbox includes surrounding context)


## 3 · Load LCMS Data

LCMS v2025-11 covers 1985–2025 (41 years). Each image in the collection represents one calendar year. We filter to the McDonald-Dunn bounding box and inspect what's available.

For the Oregon Coast Range, the LCMS `Change` band classes most relevant to this analysis are:

| Change Class | Ecological Meaning |
|---|---|
| **Tree Removal** | Timber harvest or clearing — the primary disturbance of interest; expect this to appear in many years across different compartments in an actively managed forest |
| **Vegetation Successional Growth** | Forest recovery — canopy closure and stand development after harvest; the positive counterpart to Tree Removal in the rotation cycle |
| **Wildfire** | Fire disturbance — relatively rare in the wet Coast Range but possible during drought years |
| **Stable** | No detected change — mature stands and older plantations hold this class between harvests |

> 🔍 The cell below prints the full class-value mapping so you can see the exact numeric IDs used in your LCMS version. These values are read directly from the image metadata rather than hardcoded.

In [11]:
LCMS_ASSET_FOR_PROPERTIES = 'USFS/GTAC/LCMS/v2024-10'
LCMS_ASSET = 'projects/gtac-data-publish/assets/LCMS/Product_Version/2025-11'

# Filter to study area
lcms = ee.ImageCollection(LCMS_ASSET_FOR_PROPERTIES).filterBounds(study_area)

# Basic metadata
n_images = lcms.size().getInfo()
years    = lcms.aggregate_array('year').distinct().sort().getInfo()
bands    = lcms.first().bandNames().getInfo()

print(f'Images in collection : {n_images}')
print(f'Years                : {years[0]}–{years[-1]}')
print(f'Bands                : {bands}')

# ── Print Change class mapping ──────────
# Change class mapping is stored in the older LCMS asset, which is used for metadata only.
sample_img    = ee.ImageCollection(LCMS_ASSET_FOR_PROPERTIES).first()
change_names  = sample_img.get('Change_class_names').getInfo()
change_values = sample_img.get('Change_class_values').getInfo()
change_palettes = sample_img.get('Change_class_palette').getInfo()

# # ── Apply 2024-10 symbology to the 2025 collection ────────────────────────────
# from geeViz.examples.lcmsLevelLookup import getLevelNRemap, all_lookup_2024_10


# # Build a single props dict for all three bands 
# # Uses deepest level available in the lookup for each band
# viz_props = {}
# for band in ['Land_Cover', 'Land_Use', 'Change']:
#     level = max(len(k.split('-')) for k in all_lookup_2024_10[band].keys())
#     viz_props.update(getLevelNRemap(level, band, all_lookup_2024_10)['viz_dict'])

# # Stamp the props onto every image in the 2025 collection
# lcms = lcms.map(lambda img: img.set(viz_props))

# Inspect the properties of the first image in the collection
change_img = lcms.first().select('Change')
print('Change image properties:')
for key in change_img.propertyNames().getInfo():
    print(f'  {key}: {change_img.get(key).getInfo()}')

Images in collection : 40
Years                : 1985–2024
Bands                : ['Change', 'Land_Cover', 'Land_Use', 'Change_Raw_Probability_Slow_Loss', 'Change_Raw_Probability_Fast_Loss', 'Change_Raw_Probability_Gain', 'Land_Cover_Raw_Probability_Trees', 'Land_Cover_Raw_Probability_Tall-Shrubs-and-Trees-Mix', 'Land_Cover_Raw_Probability_Shrubs-and-Trees-Mix', 'Land_Cover_Raw_Probability_Grass-Forb-Herb-and-Trees-Mix', 'Land_Cover_Raw_Probability_Barren-and-Trees-Mix', 'Land_Cover_Raw_Probability_Tall-Shrubs', 'Land_Cover_Raw_Probability_Shrubs', 'Land_Cover_Raw_Probability_Grass-Forb-Herb-and-Shrubs-Mix', 'Land_Cover_Raw_Probability_Barren-and-Shrubs-Mix', 'Land_Cover_Raw_Probability_Grass-Forb-Herb', 'Land_Cover_Raw_Probability_Barren-and-Grass-Forb-Herb-Mix', 'Land_Cover_Raw_Probability_Barren-or-Impervious', 'Land_Cover_Raw_Probability_Snow-or-Ice', 'Land_Cover_Raw_Probability_Water', 'Land_Use_Raw_Probability_Agriculture', 'Land_Use_Raw_Probability_Developed', 'Land_Use_Raw_Prob

## 4 · Interactive Map — Current Forest State and Harvest History

The map below shows three layers for the McDonald-Dunn Research Forest:

1. **Land Cover 2025** — what is on the ground today; the active rotation management creates a striking patchwork of Trees, Shrubs, and early-seral vegetation
2. **Most Common Change Agent (1985–2025)** — the modal LCMS Change class for each pixel across the full record
3. **Ever Harvested 1985–2025** — pixels flagged as "Tree Removal" in *any* year of the 41-year LCMS record

The **Ever Harvested** layer will show a much larger and more spatially distributed pattern than at a comparable LTER or wilderness site — the expected signature of an actively managed teaching forest where harvests occur across multiple compartments every decade.

> 🗺️ Toggle layers on/off with the layer panel on the right. Click any pixel to query its class value. Draw a polygon and click "Chart Selected Area" to analyze custom sub-regions.

In [13]:
Map.clearMap()

lcms_most_recent = lcms.filter(ee.Filter.eq('year', END_YEAR))

# Land Cover 2025
Map.addLayer(
    lcms_most_recent.select('Land_Cover'),
    {'autoViz': True, 'canAreaChart': True},
    f'Land Cover {END_YEAR}',
    True,
)

# Most recent change
lcms_most_recent_change = lcms_most_recent.select('Change').first().clip(study_area)
Map.addLayer(
    lcms_most_recent_change,
    {'autoViz': True, 'canAreaChart': True},
    f'Most Recent Change Agent ({END_YEAR})',
    True,
)

# Most common Change agent across the full record
lcms_most_common_change = ee.Image(lcms.select('Change').mode()).clip(study_area)
print(lcms_most_common_change.getInfo())
Map.addLayer(
    lcms_most_common_change,
    {'autoViz': True, 'canAreaChart': True},
    'Most Common Change Agent (1985-2025)',
    False,
)

# ── Dynamically find the Tree Removal class value from image metadata ──────────
name_to_val = dict(zip(change_names, change_values))
TREE_REMOVAL_VAL = name_to_val.get(
    'Tree Removal',
    name_to_val.get('Non-Fire Mechanical', 5),
)
print(f'Tree Removal class value in this LCMS version: {TREE_REMOVAL_VAL}')

# ── "Ever Harvested" — any pixel with Tree Removal detected in any LCMS year ──
ever_harvested = (
    lcms.select('Change')
    .map(lambda img: img.eq(TREE_REMOVAL_VAL))
    .max()
    .selfMask()
    .rename('ever_harvested')
)
ever_harvested = ever_harvested.set({
    'ever_harvested_class_values':  [1],
    'ever_harvested_class_names':   ['Tree Removal detected (any year 1985-2025)'],
    'ever_harvested_class_palette': ['c47b1e'],
})
Map.addLayer(ever_harvested, {'autoViz': True}, 'Ever Harvested 1985-2025', True)

# Study area bounding box outline
Map.addLayer(
    ee.Feature(study_area, {}),
    {'layerType': 'geeVector', 'strokeColor': 'ffff00', 'strokeWidth': 2.5,
     'fillColor': '00000000'},
    'McDonald-Dunn BBox',
    True,
)

Map.setCenter(-123.34, 44.70, 11)
Map.view()


Adding layer: Land Cover 2024
Adding layer: Most Recent Change Agent (2024)
{'type': 'Image', 'bands': [{'id': 'Change', 'data_type': {'type': 'PixelType', 'precision': 'int', 'min': 0, 'max': 255}, 'dimensions': [1, 1], 'origin': [-124, 44], 'crs': 'EPSG:4326', 'crs_transform': [1, 0, 0, 0, 1, 0]}], 'properties': {'system:footprint': {'geodesic': False, 'type': 'Polygon', 'coordinates': [[[-123.45999999999998, 44.6], [-123.22, 44.6], [-123.22, 44.82], [-123.45999999999998, 44.82], [-123.45999999999998, 44.6]]]}}}
Adding layer: Most Common Change Agent (1985-2025)
Tree Removal class value in this LCMS version: 9
Adding layer: Ever Harvested 1985-2025
Adding layer: McDonald-Dunn BBox
Starting webmap
Using eeCreds proxy at http://127.0.0.1:8890/ee-api (creds=ee-persistent)
geeView URL: http://localhost:8080/geeView/?v=1784297880725


## 5 · Harvest Comparison Polygons

We define four representative management zones spanning different harvest eras within McDonald-Dunn, forming a natural chronosequence from oldest to most recently cut. Each polygon captures a distinct cohort in the rotation cycle.

| Polygon | Approx. Last Harvest | Age at LCMS Start (1985) | Expected LCMS Signal |
|---------|---------------------|--------------------------|---------------------|
| **Southern McDonald (older stands)** | ~1955–1965 | ~20–30 yrs | Canopy already closing at LCMS start; Trees % high throughout; "Stable" dominant in Change |
| **Central McDonald (~1982–1988)** | ~1982–1988 | Harvested at LCMS onset | Tree Removal spike in early Change record; Shrub peak → Trees recovery through 1990s–2000s |
| **Western McDonald (~1998–2006)** | ~1998–2006 | Mid-record | Tree Removal spike mid-LCMS; Grass → Shrub → early Trees visible |
| **Dunn Forest (~2010–2018)** | ~2010–2018 | Recent | Tree Removal near end of record; Shrub/Grass dominant; Trees % rising toward 2025 |

Together, these four areas span roughly 55 years of the harvest-to-recovery cycle — the same trajectory at four different lags through the LCMS era. In contrast to a protected research forest, **some older polygons may show a second Tree Removal event** if a second rotation was completed within the 40-year window, producing a distinctive "harvest → recovery → harvest again" double-spike signature.

> ⚠️ These polygons are **approximate** and intended for comparative education. Adjust the `BBox` coordinates to match verified harvest unit shapefiles from the OSU Research Forest Business Office or the [2025 Management Plan](https://cf.forestry.oregonstate.edu/our-forests/2025-mcdonald-dunn-forest-plan).

In [14]:
# ── Define harvest comparison areas ───────────────────────────────────────────
# Keys = display labels; values = approximate bounding boxes.
# Adjust coordinates to match verified harvest unit shapefiles from the
# OSU Research Forest Business Office at Peavy Arboretum.
harvest_areas = {
    'Southern McDonald (clearcut ~1955-1965)':  ee.Geometry.BBox(-123.37, 44.62, -123.29, 44.67),
    'Central McDonald (harvest ~1982-1988)':    ee.Geometry.BBox(-123.41, 44.65, -123.33, 44.71),
    'Western McDonald (harvest ~1998-2006)':    ee.Geometry.BBox(-123.45, 44.66, -123.38, 44.73),
    'Dunn Forest (harvest ~2010-2018)':         ee.Geometry.BBox(-123.38, 44.73, -123.26, 44.80),
}

# Quick area check
print('Harvest polygon areas:')
for name, geom in harvest_areas.items():
    area = geom.area(maxError=100).divide(1e6).getInfo()
    print(f'  {name:50s}: {area:.2f} km2')

# ── Colors for each polygon ────────────────────────────────────────────────────
POLY_COLORS = {
    'Southern McDonald (clearcut ~1955-1965)':  'e8a838',
    'Central McDonald (harvest ~1982-1988)':    'e86038',
    'Western McDonald (harvest ~1998-2006)':    '38a8e8',
    'Dunn Forest (harvest ~2010-2018)':         '38e878',
}

# ── Add polygon outlines to the existing map from Section 4 ───────────────────
for name, geom in harvest_areas.items():
    c = POLY_COLORS[name]
    Map.addLayer(
        ee.Feature(geom, {}),
        {'layerType': 'geeVector', 'strokeColor': c, 'strokeWidth': 3,
         'fillColor': c + '35'},
        name,
    )

Map.setCenter(-123.34, 44.70, 11)
Map.view()


Harvest polygon areas:
  Southern McDonald (clearcut ~1955-1965)           : 35.19 km2
  Central McDonald (harvest ~1982-1988)             : 42.20 km2
  Western McDonald (harvest ~1998-2006)             : 43.07 km2
  Dunn Forest (harvest ~2010-2018)                  : 73.74 km2
Adding layer: Southern McDonald (clearcut ~1955-1965)
Adding layer: Central McDonald (harvest ~1982-1988)
Adding layer: Western McDonald (harvest ~1998-2006)
Adding layer: Dunn Forest (harvest ~2010-2018)
Starting webmap
Using eeCreds proxy at http://127.0.0.1:8890/ee-api (creds=ee-persistent)
geeView URL: http://localhost:8080/geeView/?v=1784297906409


## 6 · Recovery Trajectories — Land Cover Through Time

For each harvest area we compute the **annual percentage of each Land Cover class** from 1985 to 2025. The post-clearcut succession in Oregon Coast Range Douglas-fir forests follows a well-documented pathway:

| Stage | Dominant Cover | Typical Post-Harvest Timing |
|-------|---------------|-----------------------------|
| **Pioneer** | Grass/Forb/Herb | Years 1–3 |
| **Early shrub** | Shrubs (vine maple, red alder, bracken fern) | Years 2–10 |
| **Crown closure** | Trees (Douglas-fir canopy closing in plantations) | Years 8–20 |
| **Closed canopy** | Trees + Tall Trees and Shrubs | Years 18+ |

Coast Range forests typically transition *faster* from Shrub to Trees than comparable western Cascade plots — lower elevation, warmer summers, and reliable winter moisture accelerate plantation establishment. You may see the Shrub → Trees crossover within 10–15 years of harvest.

Where a polygon sits in this sequence **in 1985** reflects how far along recovery was when the LCMS record began. Older stand polygons (~1955–1965 cut) should open with high Trees %; more recently cut polygons should show progressively more of the Grass → Shrub → Trees arc within the chart.

> 🔍 Each chart is a stacked line plot — the height of each color band shows what percentage of the polygon was in that cover class that year. Four charts, one per management zone, let you compare recovery trajectories across the chronosequence.

In [15]:
lc_results = {}

for name, geom in harvest_areas.items():
    lcms_poly = ee.ImageCollection(LCMS_ASSET).filterBounds(geom)
    slug = (
        name[:18]
        .replace(' ', '_').replace('(', '').replace(')', '')
        .replace('~', '').replace('-', '_').strip('_')
    )
    result = cl.summarize_and_chart(
        lcms_poly,
        geometry=geom,
        band_names='Land_Cover',
        scale=30,               # 30 m — appropriate for small harvest polygons
        area_format='Percentage',
        title=f'Land Cover — {name}',
        chart_type='line',
        stacked=True,
        date_format='YYYY',
        width=950,
        height=440,
    )
    # Output results to a dictionary for later use (e.g., saving charts)
    lc_results[name] = result

    # Optionally save the chart to an html file
    # fname = f'macdunn_lc_{slug}.html'
    # cl.save_chart_html(result['chart'], fname)
    # print(f'Saved: {fname}')

    result['chart'].show()


### 6a · Inspect the Recovery Data

The `summarize_and_chart()` call returns both a chart and the raw DataFrame. Below we extract the **Trees** column for all four harvest areas and display them in a single table for direct numerical comparison.

In [17]:
import pandas as pd

# Extract the "Trees" column from each polygon's DataFrame
# Column name may be 'Trees' or include a class-value prefix like '1 — Trees'
trees_pct = {}
for name, result in lc_results.items():
    df = result['df']
    trees_col = next(
        (c for c in df.columns if 'Trees' in c and 'Tall' not in c),
        None,
    )
    if trees_col:
        trees_pct[name] = df[trees_col].round(1)
    else:
        print(f'Warning: Trees column not found for "{name}". Columns: {df.columns.tolist()}')

if trees_pct:
    trees_df = pd.DataFrame(trees_pct)
    trees_df.index.name = 'Year'
    print('Annual Trees cover (%) by harvest area:\n')
    print(trees_df.to_markdown())


Annual Trees cover (%) by harvest area:

|   Year |   Southern McDonald (clearcut ~1955-1965) |   Central McDonald (harvest ~1982-1988) |   Western McDonald (harvest ~1998-2006) |   Dunn Forest (harvest ~2010-2018) |
|-------:|------------------------------------------:|----------------------------------------:|----------------------------------------:|-----------------------------------:|
|   1985 |                                      96.5 |                                    94.3 |                                    69.4 |                               33.6 |
|   1986 |                                      96.2 |                                    93.8 |                                    69.2 |                               34.1 |
|   1987 |                                      96.5 |                                    94.3 |                                    70.7 |                               34.7 |
|   1988 |                                      96.4 |                         

## 7 · Change Agent Signatures — Tree Removal and Successional Growth

The LCMS `Change` band is the direct record of *what happened* in a pixel each year. For an actively managed forest like McDonald-Dunn we expect Tree Removal events to appear across *multiple* decades — not just once — reflecting the rotation harvest cycle. Each removal event should be followed by a recovery sequence in the Change band.

**What to expect by polygon:**

| Polygon | Tree Removal in LCMS? | Successional Growth? |
|---------|----------------------|----------------------|
| Southern McDonald (~1955–1965) | Unlikely — harvest predates 1985; watch for a *second* removal event if this stand was re-harvested ~2000–2015 | Possible early in record as 1960s plantation matures; absent after crown closure |
| Central McDonald (~1982–1988) | Yes — spike visible near LCMS onset | Yes — follows within ~3–7 years of the removal |
| Western McDonald (~1998–2006) | Yes — spike mid-record | Yes — increasingly dominant through 2010s–2020s |
| Dunn Forest (~2010–2018) | Yes — spike near end of record | Emerging — recent enough that Trees % is still low toward 2025 |

**Also watch for repeat disturbance**: Coast Range commercial rotation ages can be as short as 35–50 years. If any of the older polygons were re-harvested within the LCMS window, you will see a second Tree Removal spike after a period of Successional Growth — a "harvest → recovery → harvest again" signature that is only visible in an actively managed forest, not at an LTER or wilderness site.

> 🔍 **Look for the lag:** How many years after Tree Removal does Successional Growth become the dominant change class? Coast Range forests typically complete this transition within 3–7 years of harvest.

In [18]:
change_results = {}

for name, geom in harvest_areas.items():
    lcms_poly = ee.ImageCollection(LCMS_ASSET_FOR_PROPERTIES).filterBounds(geom)
    slug = (
        name[:18]
        .replace(' ', '_').replace('(', '').replace(')', '')
        .replace('~', '').replace('-', '_').strip('_')
    )
    result = cl.summarize_and_chart(
        lcms_poly,
        geometry=geom,
        band_names='Change',
        scale=30,
        area_format='Percentage',
        title=f'Change Agents — {name}',
        chart_type='line',
        stacked=False,          # unstacked so individual classes are legible
        date_format='YYYY',
        width=950,
        height=440,
    )
    change_results[name] = result
    #fname = f'macdunn_change_{slug}.html'
    #cl.save_chart_html(result['chart'], fname)
    #print(f'Saved: {fname}')
    result['chart'].show()


## 8 · Combined Comparison — Tree Cover Recovery Across All Management Zones

Now we overlay the **Trees cover percentage** from all four management zones on a single chart. This is the key comparison plot that directly answers the research question about how the Coast Range harvest-to-recovery cycle appears across 40 years of LCMS data:

- **Oldest stands** (Southern McDonald) should show relatively high, stable Trees percentages throughout, as regeneration from 1950s–1960s cuts was well advanced before 1985.
- **Early-record harvest** (Central McDonald, ~1982–1988) should show a dip from Trees at the LCMS onset followed by a steady rise as the plantation closes the canopy.
- **Mid-record harvest** (Western McDonald, ~1998–2006) should show the lowest Trees values in the early 2000s, then a steep climb as young Douglas-fir achieves crown closure through the 2010s–2020s.
- **Most recent harvest** (Dunn Forest, ~2010–2018) should show the most recent Tree Removal dip and the earliest stages of recovery still underway in 2025.

**Watch for:** If the oldest polygon shows a *second* dip in Trees%, that is the rotation harvest cycle in action — a signature uniquely visible in an actively managed forest. A wide vertical spread between curves in 2025 confirms that LCMS successfully resolves the age-class mosaic of the rotation system.

> 📈 If a polygon shows unexpectedly low Trees% for an older harvest era, consider whether the bounding box captures mixed land cover (roads, riparian corridors, non-forest patches) that dilute the tree-cover signal.

In [ ]:
import plotly.graph_objects as go

# lc_results is built in cell 6 (Section 6 code cell); re-run that cell if needed.

LINE_COLORS = {
    'Southern McDonald (clearcut ~1955-1965)':  '#e8a838',
    'Central McDonald (harvest ~1982-1988)':    '#e86038',
    'Western McDonald (harvest ~1998-2006)':    '#38a8e8',
    'Dunn Forest (harvest ~2010-2018)':         '#38e878',
}
DASH_STYLES = ['solid', 'dash', 'dot', 'dashdot']

fig = go.Figure()

for (name, result), dash in zip(lc_results.items(), DASH_STYLES):
    df = result['df']
    trees_col = next(
        (c for c in df.columns if 'Trees' in c and 'Tall' not in c),
        None,
    )
    if trees_col:
        fig.add_trace(go.Scatter(
            x=df.index,
            y=df[trees_col].values,
            name=name,
            mode='lines+markers',
            line=dict(color=LINE_COLORS[name], width=2.5, dash=dash),
            marker=dict(size=4),
        ))

fig.update_layout(
    title='McDonald-Dunn Research Forest — Tree Cover Recovery by Management Zone 1985-2025',
    xaxis=dict(title='Year', tickmode='linear', dtick=5),
    yaxis=dict(title='Trees (% of polygon area)', range=[0, 100]),
    legend=dict(orientation='h', yanchor='top', y=-0.20, xanchor='left', x=0),
    width=1050,
    height=550,
    template='plotly_white',
)

fig.write_html('macdunn_recovery_comparison.html')
print('Comparison chart saved to macdunn_recovery_comparison.html')
fig.show()


## 9 · Land Use Transitions — Sankey Diagram

The Land Use Sankey shows how land area has shifted between use classes across the whole study area at key time steps. For McDonald-Dunn — unlike a protected or wilderness site — the Forest class should remain the dominant category throughout, with relatively small flows that reflect active management decisions rather than broad land-use conversion.

The transition years below align with management and policy inflection points:
- **1985** → LCMS baseline; active commercial harvest throughout both blocks
- **1995** → post-1994 Northwest Forest Plan — **note that McDonald-Dunn is OSU land, not USFS, so it was unaffected**; contrast with what a comparable Siuslaw NF analysis would show
- **2010** → updated OSU management plan era; diverse age classes across the forest
- **2025** → present; ongoing active management with recent harvests visible in the LCMS record

> 💡 **The interesting contrast**: Adjacent USFS lands (Siuslaw, Willamette NFs) would show large flows *out of Forest land use* after the 1994 Northwest Forest Plan. McDonald-Dunn, as a university forest, continued active management — the Sankey may show a stable Forest block throughout, making the policy boundary literally visible on a map.

In [ ]:
lu_sankey = cl.summarize_and_chart(
    lcms,
    geometry=study_area,
    band_names='Land_Use',
    scale=120,               # coarser scale acceptable for the full study area
    area_format='Percentage',
    title='McDonald-Dunn Research Forest — Land Use Transitions 1985 -> 1995 -> 2010 -> 2025',
    sankey=True,
    transition_periods=[1985, 1995, 2010, 2025],
    min_percentage=0.5,      # hide flows < 0.5% to keep the diagram readable
    width=1000,
    height=600,
)

cl.save_chart_html(lu_sankey['chart'], 'macdunn_land_use_sankey.html')
print('Sankey saved to macdunn_land_use_sankey.html')

display(HTML(lu_sankey['chart']))


### 9a · Land Use Transition Matrices

The raw transition numbers behind the Sankey — useful for quantifying exactly how much area moved between classes in each period.

In [ ]:
if 'matrix' in lu_sankey:
    for period_key, mat in lu_sankey['matrix'].items():
        print(f'### {period_key}')
        print(mat.to_markdown())
        print()


## 10 · Interactive Time-Lapse — 40 Years of Change and Recovery

The time-lapse steps through each year's LCMS Change and Land Cover bands, letting you see *where* Tree Removal and recovery occurred spatially across the McDonald-Dunn landscape — not just *how much* area they affected. The management zone outlines are overlaid as reference.

**What to look for:**
- **Scattered Tree Removal events** appearing in *different* compartments across different years — the spatial mosaic of a rotation management system, fundamentally different from a single large wildfire or clearcut
- The **Shrub → Trees** transition progressing through each harvest unit at different times, reflecting the age-class mosaic built up by decades of rotation harvest
- **Repeat disturbance** in older polygons: if Tree Removal appears twice in the same location decades apart, that is the rotation harvest cycle completing a second pass
- Adjacent unharvested old-growth remnants or long-uncut patches that hold "Stable" throughout, providing visual contrast to the actively managed blocks

> ⏱️ This may take 30–60 seconds to load — it is rendering 41 annual layers.

In [ ]:
Map.clearMap()

# Annual LCMS Change time-lapse (slider in geeViz map)
Map.addTimeLapse(
    lcms.select('Change'),
    {'autoViz': True, 'canAreaChart': True},
    'Change Agent (Annual)',
    visible=True,
)

# Annual Land Cover time-lapse
Map.addTimeLapse(
    lcms.select('Land_Cover'),
    {'autoViz': True, 'canAreaChart': True},
    'Land Cover (Annual)',
    visible=False,
)

# Management zone outlines for spatial reference
for name, geom in harvest_areas.items():
    c = POLY_COLORS[name]
    Map.addLayer(
        ee.Feature(geom, {}),
        {'layerType': 'geeVector', 'strokeColor': c, 'strokeWidth': 3,
         'fillColor': '00000000'},
        name,
        False,
    )

# Study area outline
Map.addLayer(
    ee.Feature(study_area, {}),
    {'layerType': 'geeVector', 'strokeColor': 'ffffff', 'strokeWidth': 2,
     'fillColor': '00000000'},
    'McDonald-Dunn Study Area',
)

Map.setCenter(-123.34, 44.70, 11)
Map.view()


## 11 · Bonus — Classify Recovery Stages Across the Study Area

Rather than looking at individual polygons, here we classify *every pixel* in the study area by how many years (out of 41) it was mapped as Trees. This "years-as-trees" metric is a proxy for successional maturity — but in an actively managed forest it also maps directly onto the rotation age-class structure: a pixel that scores 40+ has been in continuous tree cover (old plantation or old-growth remnant); a pixel that scores 5 was recently clearcut or is perpetually in a non-tree cover class.

| Stage | Years as Trees | Interpretation for McDonald-Dunn |
|-------|---------------|----------------------------------|
| **Early seral** | 0–5 | Recently clearcut or non-forested; grass or shrub dominant |
| **Mid seral** | 6–15 | Young plantation establishing; canopy not yet closed |
| **Late seral** | 16–29 | Dense young forest; typical rotation harvest age class |
| **Mature/established stand** | 30–41 | Closed-canopy throughout nearly the entire record; 1950s–1960s plantations or old-growth remnants |

In an actively managed forest the spatial pattern of these stages should map closely onto the documented harvest rotation mosaic — each management compartment should cluster in a recovery stage reflecting when it was last cut. This makes "years-as-trees" a simple but powerful visual summary of the management age-class structure across the entire forest.

In [ ]:
Map.clearMap()

# ── Count how many years each pixel was classified as Trees (class 1) ──────────
years_as_trees = (
    lcms.select('Land_Cover')
    .map(lambda img: img.eq(1).rename('is_trees'))
    .sum()
    .rename('years_as_trees')
)

# ── Classify into recovery / age-class stages ──────────────────────────────────
# Start all pixels at 1 (Early seral), then override upward as thresholds are met.
recovery_stage = (
    ee.Image(1)
    .where(years_as_trees.gt(5),  2)   # Mid seral
    .where(years_as_trees.gt(15), 3)   # Late seral
    .where(years_as_trees.gt(29), 4)   # Mature/established stand
    .rename('recovery_stage')
    .updateMask(years_as_trees.gte(0)) # keep same spatial extent
)

recovery_stage = recovery_stage.set({
    'recovery_stage_class_values':  [1, 2, 3, 4],
    'recovery_stage_class_names':   [
        'Early seral (0-5 yrs Trees)',
        'Mid seral (6-15 yrs Trees)',
        'Late seral (16-29 yrs Trees)',
        'Mature/established stand (30-41 yrs Trees)',
    ],
    'recovery_stage_class_palette': ['f7dc6f', 'e67e22', '27ae60', '1a5276'],
})

# ── Add layers ────────────────────────────────────────────────────────────────
Map.addLayer(
    lcms_most_recent.select('Land_Cover'),   # defined in Section 4 code cell
    {'autoViz': True, 'canAreaChart': True},
    f'Land Cover {END_YEAR}',
    False,
)
Map.addLayer(
    recovery_stage,
    {'autoViz': True, 'canAreaChart': True},
    'Recovery Stage (years as Trees, 1985-2025)',
    True,
)

# Management zone outlines
for name, geom in harvest_areas.items():
    c = POLY_COLORS[name]
    Map.addLayer(
        ee.Feature(geom, {}),
        {'layerType': 'geeVector', 'strokeColor': c, 'strokeWidth': 3,
         'fillColor': '00000000'},
        name,
        True,
    )

Map.addLayer(
    ee.Feature(study_area, {}),
    {'layerType': 'geeVector', 'strokeColor': 'ffffff', 'strokeWidth': 2,
     'fillColor': '00000000'},
    'McDonald-Dunn Study Area',
)

Map.setCenter(-123.34, 44.70, 11)
Map.view()


## 12 · Key Takeaways and Next Steps

### What LCMS tells us about an actively managed Oregon Coast Range research forest

After running this notebook you should be able to answer:

1. **Harvest detection** — Does LCMS reliably detect the Tree Removal events in the McDonald-Dunn management zones? Which silvicultural treatments (clearcut vs. shelterwood vs. selection) produce the clearest LCMS signal? Do smaller selection harvests appear as reduced Tree Removal intensity rather than a distinct spike?

2. **Recovery rate** — How many years does it take for the Trees land-cover class to re-dominate each polygon after harvest? Is the Coast Range faster than comparable western Cascades analyses, as expected given the lower-elevation climate?

3. **Rotation visibility** — Can you identify a second Tree Removal event in any of the older polygons — evidence that a second rotation was completed within the 1985–2025 LCMS window? This is the defining feature of a managed forest that would not appear in a protected LTER analysis.

4. **Chronosequence integrity** — Do the four management-zone polygons form a coherent chronosequence on the combined Trees-recovery chart (oldest at top, youngest at bottom throughout the record)? Any violations may reflect unmapped re-harvests or strong topographic/aspect effects on recovery rate.

5. **Land use stability** — Does the Sankey show the Forest land-use class holding stable throughout, consistent with continuous OSU forestry management? How does this contrast with what you would expect from an adjacent USFS forest subject to the 1994 Northwest Forest Plan?

6. **Recovery stage map** — Does the "years-as-trees" spatial pattern correspond to the known management rotation mosaic? High correspondence supports using LCMS as a forest age-class mapping tool for non-federal managed forests across the Pacific Northwest.

---

### Going further

| Idea | How |
|------|-----|
| Use official harvest unit polygons | Request GIS data from the OSU Research Forest Business Office at Peavy Arboretum, or see the [2025 Management Plan](https://cf.forestry.oregonstate.edu/our-forests/2025-mcdonald-dunn-forest-plan) |
| Compare with adjacent USFS land | Replace `study_area` with `sal.getUSFSForests(forest_name='Siuslaw')` to see the 1994 Northwest Forest Plan management shift in contrast to OSU's continuous management |
| Add topographic controls | Load `USGS/SRTMGL1_003`, compute slope and aspect; compare Trees% recovery rate by topographic position across Coast Range terrain |
| Extend to drier eastern Oregon | Duplicate the notebook with a Malheur or Ochoco NF bounding box — drier climate, fire-driven disturbance, much slower recovery |
| Overlay MTBS fire perimeters | Load MTBS from the GEE catalog — while the Coast Range is wet, the 2020 Labor Day fires burned surprisingly close; check whether any perimeter intersects the study area |
| Validate harvest detection | If you have official harvest record shapefiles, overlay them on the "Ever Harvested" layer to compute LCMS detection rates by silvicultural treatment type |
| Export annual land cover maps | Use the EE batch export API (`ee.batch.Export.image.toDrive(...)`) for GeoTIFF outputs suitable for GIS integration with the OSU forest management records |

---

### Data citation

> USFS GTAC. (2025). *Landscape Change Monitoring System v2025-11*. USDA Forest Service, Geospatial Technology and Applications Center. [https://www.fs.usda.gov/lcms](https://www.fs.usda.gov/lcms)

> Google Earth Engine catalog: `projects/gtac-data-publish/assets/LCMS/Product_Version/2025-11`

> McDonald-Dunn Research Forest, Oregon State University College of Forestry. [https://cf.forestry.oregonstate.edu/our-forests/mcdonald-and-dunn-forests](https://cf.forestry.oregonstate.edu/our-forests/mcdonald-and-dunn-forests)